In [1]:
import sys
# Add the desired directory to sys.path
sys.path.append('build/')

import randSVD
from sklearn.utils.extmath import randomized_svd

seed = 7050
tol = 1e-3

## Test matrix

In [2]:
import numpy as np

class SymmetricTestMatrix:
    def __init__(self, size, eig_val_decay):
        self.rows = size
        self.cols = size
        self.eig_val_decay = eig_val_decay
        
        # Calculate rank
        self.rank = size
        
        # Generate random U matrix
        self.U = np.random.rand(size, self.rank)
        
        # Orthogonalize U using QR decomposition
        self.U, _ = np.linalg.qr(self.U)
        
        # Set the singular values based on the decay option
        if self.eig_val_decay == "fast":
            self.eig_vals = np.power(0.95, np.arange(self.rank))
        else:
            self.eig_vals = 1/np.log(2 + np.arange(self.rank))

        self.A = (self.U @ np.diag(self.eig_vals) @ self.U.T).astype(np.float64)
        
    def matrixU(self):
        return self.U
    
    def eigenValues(self):
        return self.eig_vals
    
    def matrixA(self):
        return self.A
    

## Error metrics

In [3]:
def compute_errors(test_matrix,U,sing_vals,V):
    errors = {}

    rank = sing_vals.size

    errors["rec_error"] = np.linalg.norm(test_matrix.matrixA() - U@np.diag(sing_vals)@V.T, 'fro')
    errors["eig_val_error"] = np.linalg.norm(test_matrix.eigenValues()[:rank] - sing_vals)
    errors["eig_vect_error"] = min(
        max(np.minimum(np.linalg.norm(test_matrix.matrixU()[:,:rank]-U,axis=0),np.linalg.norm(test_matrix.matrixU()[:,:rank]+U,axis=0))),
        max(np.minimum(np.linalg.norm(test_matrix.matrixU()[:,:rank]-V,axis=0),np.linalg.norm(test_matrix.matrixU()[:,:rank]+V,axis=0)))
    )
    
    
    return errors

## Computations

In [5]:
import time
import pandas as pd

# Assuming SymmetricTestMatrix, compute_errors, randSVD, etc., are already defined

test_results = pd.DataFrame(columns=['svd', 'size', 'decay', 'replica', 'rec_error', 'eig_val_error', 'eig_vect_error', 'execution_time'])
tested_sizes = [1000,2000,3000,4000,5000]

rank = 5  # to compare with rbki
n_iter = 10
tol = 1e-16
seed = 7050

n_replicas = 1

rsi = randSVD.RSI(seed, tol)
rbki = randSVD.RBKI(seed, tol)
nys_rsi = randSVD.NysRSI(seed, tol)
nys_rbki = randSVD.NysRBKI(seed, tol)

for test_size in tested_sizes:
    print("Size: ", test_size)
    for decay in ['fast', 'slow']:
        print("Decay:", decay)
        test_matrix = SymmetricTestMatrix(test_size, decay)

        # Exact SVD computation
        start_time = time.perf_counter()
        U, S, Vh = np.linalg.svd(test_matrix.matrixA(), full_matrices=False)
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        optimal_error = compute_errors(test_matrix=test_matrix, U=U[:, :rank], sing_vals=S[:rank], V=Vh.T[:, :rank])

        test_results = pd.concat([test_results, pd.DataFrame([["exact", test_size, decay, 1, *optimal_error.values(), execution_time]], 
                                                             columns=test_results.columns)], ignore_index=True)

        for i in range(1, n_replicas + 1):

            # RSI computation
            start_time = time.perf_counter()
            rsi.compute(test_matrix.matrixA(), rank, n_iter)
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            errors_rsi = compute_errors(test_matrix=test_matrix, U=rsi.matrixU(), sing_vals=rsi.singularValues(), V=rsi.matrixV())
            test_results = pd.concat([test_results, pd.DataFrame([["rsi", test_size, decay, i, *errors_rsi.values(), execution_time]], 
                                                                 columns=test_results.columns)], ignore_index=True)

            # RBKI computation
            start_time = time.perf_counter()
            rbki.compute(test_matrix.matrixA(), rank, n_iter)
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            errors_rbki = compute_errors(test_matrix=test_matrix, U=rbki.matrixU(), sing_vals=rbki.singularValues(), V=rbki.matrixV())
            test_results = pd.concat([test_results, pd.DataFrame([["rbki", test_size, decay, i, *errors_rbki.values(), execution_time]], 
                                                                 columns=test_results.columns)], ignore_index=True)

            # NysRSI computation
            start_time = time.perf_counter()
            nys_rsi.compute(test_matrix.matrixA(), rank, 2 * n_iter + 1)
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            errors_nys_rsi = compute_errors(test_matrix=test_matrix, U=nys_rsi.matrixU(), sing_vals=nys_rsi.eigenValues(), V=nys_rsi.matrixU())
            test_results = pd.concat([test_results, pd.DataFrame([["nys_rsi", test_size, decay, i, *errors_nys_rsi.values(), execution_time]], 
                                                                 columns=test_results.columns)], ignore_index=True)

            # NysRBKI computation
            start_time = time.perf_counter()
            nys_rbki.compute(test_matrix.matrixA(), rank, 2 * n_iter)
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            errors_nys_rbki = compute_errors(test_matrix=test_matrix, U=nys_rbki.matrixU(), sing_vals=nys_rbki.eigenValues(), V=nys_rbki.matrixU())
            test_results = pd.concat([test_results, pd.DataFrame([["nys_rbki", test_size, decay, i, *errors_nys_rbki.values(), execution_time]], 
                                                                 columns=test_results.columns)], ignore_index=True)

            # Scikit-learn computation (randomized SVD)
            start_time = time.perf_counter()
            U, s, Vh = randomized_svd(test_matrix.matrixA(),
                                       n_components=rank,
                                       n_oversamples=rank,
                                       n_iter=n_iter,
                                       power_iteration_normalizer='QR',
                                       transpose=False,
                                       random_state=seed)
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            errors_sklearn = compute_errors(test_matrix=test_matrix, U=U, sing_vals=s, V=Vh.T)
            test_results = pd.concat([test_results, pd.DataFrame([["scikit-learn", test_size, decay, i, *errors_sklearn.values(), execution_time]], 
                                                                 columns=test_results.columns)], ignore_index=True)

Size:  1000
Decay: fast


/var/folders/7g/9rtz76hn4j5_ws3_z5kszxk40000gn/T/ipykernel_46054/2966613017.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  test_results = pd.concat([test_results, pd.DataFrame([["exact", test_size, decay, 1, *optimal_error.values(), execution_time]],


Decay: slow
Size:  2000
Decay: fast
Decay: slow
Size:  3000
Decay: fast
Decay: slow
Size:  4000
Decay: fast
Decay: slow
Size:  5000
Decay: fast
Decay: slow


In [6]:
print(test_results)

             svd  size decay replica  rec_error  eig_val_error  \
0          exact  1000  fast       1   2.478082   1.391105e-15   
1            rsi  1000  fast       1   2.478087   1.347742e-05   
2           rbki  1000  fast       1   2.478082   4.749938e-14   
3        nys_rsi  1000  fast       1   2.478087   1.935398e-05   
4       nys_rbki  1000  fast       1   2.478082   2.137559e-13   
5   scikit-learn  1000  fast       1   2.478082   5.805173e-07   
6          exact  1000  slow       1   5.643737   2.497385e-15   
7            rsi  1000  slow       1   5.643737   5.461303e-08   
8           rbki  1000  slow       1   5.643737   6.109153e-14   
9        nys_rsi  1000  slow       1   5.643737   8.802942e-08   
10      nys_rbki  1000  slow       1   5.643737   2.241125e-13   
11  scikit-learn  1000  slow       1   5.643737   3.198870e-09   
12         exact  2000  fast       1   2.478082   2.368187e-15   
13           rsi  2000  fast       1   2.478082   5.789911e-07   
14        

In [7]:
test_results.to_csv('results/randEVD_comparison.csv', index=False)